In [14]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import yaml


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================================================
# 0. パス設定（整理版：ziparea）
# =========================================================
from pathlib import Path

CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    BASE = Path(config["data_root"]).expanduser()
else:
    BASE = Path("~/Desktop/Research/Data").expanduser()
    
SMALLAREA_PATH = BASE / "Processed" / "Census2020" / "smallarea" / "smallarea_adi_analysis.gpkg"

ZIP_PATH = (
    BASE / "Raw" / "Geospatial" / "ZIP_polygon" / "2022"
    / "2022_GEOSPACE郵便番号ポリゴン" / "shape" / "GSP99.shp"
)

# 👇 zip専用フォルダ
OUT_DIR = BASE / "Processed" / "Census2020" / "ziparea"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 出力ファイル（命名整理）
# =========================================================

# full版（全郵便番号）
OUT_FULL_CSV = OUT_DIR / "zip_adi_full.csv"
OUT_FULL_GPKG = OUT_DIR / "zip_adi_full.gpkg"
OUT_FULL_PARQUET = OUT_DIR / "zip_adi_full.parquet"

# analysis版（ADIありのみ）
OUT_ANALYSIS_CSV = OUT_DIR / "zip_adi_analysis.csv"
OUT_ANALYSIS_GPKG = OUT_DIR / "zip_adi_analysis.gpkg"
OUT_ANALYSIS_PARQUET = OUT_DIR / "zip_adi_analysis.parquet"

# =========================================================
# チェック用（中間生成物）
# =========================================================
CHECK_DIR = BASE / "Intermediate" / "Census2020" / "zip_adi_check"
CHECK_DIR.mkdir(parents=True, exist_ok=True)

OUT_INTERSECTION = CHECK_DIR / "zip_smallarea_intersection.parquet"
OUT_WEIGHT_CHECK = CHECK_DIR / "zip_adi_weight_check.csv"

In [2]:
# =========================================================
# 1. 読み込み
# =========================================================
smallarea_gdf = gpd.read_file(SMALLAREA_PATH)
zip_gdf = gpd.read_file(ZIP_PATH)

print("smallarea_gdf:", smallarea_gdf.shape)
print("zip_gdf:", zip_gdf.shape)

print("smallarea CRS:", smallarea_gdf.crs)
print("zip CRS:", zip_gdf.crs)

print("smallarea columns:")
print(smallarea_gdf.columns.tolist())

print("zip columns:")
print(zip_gdf.columns.tolist())

smallarea_gdf: (203436, 24)
zip_gdf: (113253, 5)
smallarea CRS: EPSG:6668
zip CRS: EPSG:6668
smallarea columns:
['final_key', '市区町村コード', 'final_unit_code', 'PREF_NAME', 'CITY_NAME', '都道府県名', '市区町村名', '大字・町名', '字・丁目名', '総数', '住宅に住む一般世帯', '労働力人口', '0_総数', 'old_couple_rate', 'old_single_rate', 'single_parent_proxy_rate', 'rent_rate', 'sales_service_rate', 'agriculture_rate', 'blue_collar_rate', 'unemployment_rate', 'ADI_raw', 'exclude_nonresidential', 'geometry']
zip columns:
['郵便番号', '市区町村CD', '町域名', '管理情報', 'geometry']


In [3]:
# =========================================================
# 2. CRS を統一
# =========================================================
if zip_gdf.crs != smallarea_gdf.crs:
    zip_gdf = zip_gdf.to_crs(smallarea_gdf.crs)

print("smallarea CRS:", smallarea_gdf.crs)
print("zip CRS:", zip_gdf.crs)

smallarea CRS: EPSG:6668
zip CRS: EPSG:6668


In [4]:
# =========================================================
# 4. 必要列に絞る（整理版）
# =========================================================

zip_col = "郵便番号"  # ← 必ず実データと一致させる

# ---------------------------------------------------------
# 1. 郵便番号列を正規化（先にやる）
# ---------------------------------------------------------
zip_gdf = zip_gdf.copy()

zip_gdf[zip_col] = (
    zip_gdf[zip_col]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(7)
)

print("zip head:")
print(zip_gdf[zip_col].head(10))
print("length distribution:")
print(zip_gdf[zip_col].str.len().value_counts().sort_index())
print("unique zips:", zip_gdf[zip_col].nunique())


# ---------------------------------------------------------
# 2. 必要列の選択
# ---------------------------------------------------------
smallarea_cols = [
    "final_key",
    "ADI_raw",
    "geometry"
]
smallarea_cols = [c for c in smallarea_cols if c in smallarea_gdf.columns]

zip_cols = [zip_col, "geometry"]
zip_cols = [c for c in zip_cols if c in zip_gdf.columns]


# ---------------------------------------------------------
# 3. useデータ作成（ここで初めて作る）
# ---------------------------------------------------------
smallarea_use = smallarea_gdf[smallarea_cols].copy()
zip_use = zip_gdf[zip_cols].copy()

print("smallarea_use:", smallarea_use.shape)
print("zip_use:", zip_use.shape)

zip head:
0    0600000
1    0600001
2    0600002
3    0600003
4    0600004
5    0600005
6    0600006
7    0600007
8    0600008
9    0600009
Name: 郵便番号, dtype: object
length distribution:
郵便番号
7    113253
Name: count, dtype: int64
unique zips: 113185
smallarea_use: (203436, 3)
zip_use: (113253, 2)


In [5]:
# =========================================================
# 5. 元の小地域面積
# =========================================================
smallarea_use = smallarea_use.copy()
smallarea_use["smallarea_area"] = smallarea_use.geometry.area

print(smallarea_use[["final_key", "smallarea_area"]].head())

/var/folders/lx/_3z8wwyx3ds15_0k28hby_h80000gn/T/ipykernel_38167/3364609285.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  smallarea_use["smallarea_area"] = smallarea_use.geometry.area


      final_key  smallarea_area
0  01101_100000    2.576351e-04
1  01101_110000    2.037340e-03
2  01101_120101    1.491442e-06
3  01101_120102    7.566989e-07
4  01101_120103    7.514180e-07


In [6]:
# =========================================================
# 6. 郵便番号 × 小地域 の intersection
# =========================================================
intersection_gdf = gpd.overlay(
    zip_use,
    smallarea_use,
    how="intersection"
)

print("intersection_gdf:", intersection_gdf.shape)
print(intersection_gdf.head())

intersection_gdf: (859384, 5)
      郵便番号     final_key   ADI_raw  smallarea_area                                           geometry
0  0600000  01101_100000  4.987711        0.000258  MULTIPOLYGON (((141.30852 43.04774, 141.30853 ...
1  0600000  01101_130001  7.036658        0.000007  MULTIPOLYGON (((141.32004 43.04582, 141.31997 ...
2  0600000  01101_130002  6.230936        0.000004  POLYGON ((141.31583 43.04454, 141.31532 43.044...
3  0600000  01101_170010  4.978670        0.000006  MULTIPOLYGON (((141.31182 43.04608, 141.31171 ...
4  0600000  01101_690527  5.975916        0.000002  POLYGON ((141.31778 43.05008, 141.31812 43.049...


In [7]:
# =========================================================
# 7. 交差面積と重み
# =========================================================
intersection_gdf["intersect_area"] = intersection_gdf.geometry.area
intersection_gdf["area_weight"] = (
    intersection_gdf["intersect_area"] / intersection_gdf["smallarea_area"]
)

print(
    intersection_gdf[
        [zip_col, "final_key", "smallarea_area", "intersect_area", "area_weight", "ADI_raw"]
    ].head(20)
)

/var/folders/lx/_3z8wwyx3ds15_0k28hby_h80000gn/T/ipykernel_38167/631240768.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection_gdf["intersect_area"] = intersection_gdf.geometry.area


       郵便番号     final_key  smallarea_area  intersect_area  area_weight   ADI_raw
0   0600000  01101_100000    2.576351e-04    2.368113e-07     0.000919  4.987711
1   0600000  01101_130001    6.578482e-06    1.869124e-07     0.028413  7.036658
2   0600000  01101_130002    4.207649e-06    9.073687e-08     0.021565  6.230936
3   0600000  01101_170010    6.183223e-06    8.341725e-08     0.013491  4.978670
4   0600000  01101_690527    1.804655e-06    3.887099e-08     0.021539  5.975916
5   0600000  01101_690626    2.701867e-06    1.857392e-07     0.068745  5.202425
6   0600000  01101_690627    4.592395e-07    2.196777e-07     0.478351  4.350000
7   0600000  01101_690726    1.929344e-06    3.287561e-07     0.170398  8.498212
8   0600000  01101_690826    2.047390e-06    9.051060e-09     0.004421  6.533523
9   0600001  01101_210006    2.794467e-06    3.658102e-08     0.013091  4.419770
10  0600001  01101_210007    2.943913e-06    1.020486e-09     0.000347  6.294189
11  0600001  01101_210009   

In [8]:
# =========================================================
# 8. 面積按分したADI
# =========================================================
intersection_gdf["ADI_weighted"] = (
    intersection_gdf["ADI_raw"] * intersection_gdf["area_weight"]
)

intersection_gdf.to_parquet(OUT_INTERSECTION)
print("saved:", OUT_INTERSECTION)

saved: /Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/zip_adi_check/zip_smallarea_intersection.parquet


In [9]:
# =========================================================
# 9. 郵便番号単位に集約（加重平均に修正）
# =========================================================
zip_adi_df = (
    intersection_gdf.groupby(zip_col, dropna=False)
    .agg(
        ADI_weighted_sum=("ADI_weighted", "sum"),
        area_weight_sum=("area_weight", "sum"),
        intersect_area_sum=("intersect_area", "sum"),
        n_smallareas=("final_key", "nunique"),
    )
    .reset_index()
)

# 面積加重平均として ZIP ADI を計算
zip_adi_df["ADI_raw"] = (
    zip_adi_df["ADI_weighted_sum"] / zip_adi_df["area_weight_sum"]
)

print("zip_adi_df:", zip_adi_df.shape)
print(zip_adi_df.head())
print(zip_adi_df["ADI_raw"].describe())

zip_adi_df: (112980, 6)
      郵便番号  ADI_weighted_sum  area_weight_sum  intersect_area_sum  n_smallareas   ADI_raw
0  0000000          0.964600         0.139798            0.000006             5  6.899975
1  0010010         18.543326         3.958471            0.000007             7  4.684467
2  0010011         20.351883         4.063234            0.000008            14  5.008790
3  0010012         20.501327         3.905879            0.000007             8  5.248838
4  0010013         20.692079         3.981906            0.000007            12  5.196526
count    112980.000000
mean          7.564788
std           1.452996
min           0.751344
25%           6.674142
50%           7.453812
75%           8.322165
max          21.557795
Name: ADI_raw, dtype: float64


In [10]:
# =========================================================
# 10. 重み確認用
# =========================================================
weight_check = intersection_gdf.groupby("final_key", dropna=False).agg(
    area_weight_sum=("area_weight", "sum"),
    n_zipcodes=(zip_col, "nunique")
).reset_index()

print(weight_check.head(20))
print(weight_check["area_weight_sum"].describe())

weight_check.to_csv(OUT_WEIGHT_CHECK, index=False, encoding="utf-8-sig")
print("saved:", OUT_WEIGHT_CHECK)

       final_key  area_weight_sum  n_zipcodes
0   01101_100000              1.0          18
1   01101_110000              1.0          12
2   01101_120101              1.0           4
3   01101_120102              1.0           3
4   01101_120103              1.0           3
5   01101_120104              1.0           1
6   01101_120105              1.0           3
7   01101_120106              1.0           5
8   01101_120107              1.0           3
9   01101_120108              1.0           2
10  01101_120109              1.0           3
11  01101_120110              1.0           4
12  01101_120111              1.0           3
13  01101_120112              1.0           3
14  01101_120113              1.0           3
15  01101_120114              1.0           3
16  01101_120115              1.0           4
17  01101_120116              1.0           4
18  01101_120117              1.0           3
19  01101_120118              1.0           3
count    203436.000000
mean       

In [11]:
# =========================================================
# 11. 郵便番号ポリゴンを郵便番号単位で dissolve
# =========================================================
zip_poly = zip_use[[zip_col, "geometry"]].copy()

zip_poly = zip_poly.dissolve(
    by=zip_col,
    as_index=False,
    aggfunc="first"
)

print("zip_poly:", zip_poly.shape)
print("duplicated zip after dissolve:", zip_poly[zip_col].duplicated().sum())

# =========================================================
# 12. 郵便番号ポリゴンに結合
# =========================================================
zip_adi_gdf = zip_poly.merge(
    zip_adi_df,
    on=zip_col,
    how="left",
    validate="1:1"
)

print("zip_adi_gdf:", zip_adi_gdf.shape)
print("missing ADI_raw:", zip_adi_gdf["ADI_raw"].isna().sum())
print(zip_adi_gdf.head())

zip_poly: (113185, 2)
duplicated zip after dissolve: 0
zip_adi_gdf: (113185, 7)
missing ADI_raw: 205
      郵便番号                                           geometry  ADI_weighted_sum  area_weight_sum  intersect_area_sum  n_smallareas   ADI_raw
0  0000000  MULTIPOLYGON (((136.79269 35.25518, 136.79249 ...          0.964600         0.139798            0.000006           5.0  6.899975
1  0010010  POLYGON ((141.34908 43.07248, 141.34751 43.072...         18.543326         3.958471            0.000007           7.0  4.684467
2  0010011  POLYGON ((141.34879 43.07361, 141.34722 43.073...         20.351883         4.063234            0.000008          14.0  5.008790
3  0010012  POLYGON ((141.3485 43.07477, 141.34693 43.0745...         20.501327         3.905879            0.000007           8.0  5.248838
4  0010013  POLYGON ((141.3482 43.0759, 141.34665 43.07569...         20.692079         3.981906            0.000007          12.0  5.196526


In [12]:
# =========================================================
# 13. full版を保存
# =========================================================
zip_adi_full_gdf = zip_adi_gdf.copy()
zip_adi_full_df = pd.DataFrame(zip_adi_full_gdf.drop(columns="geometry"))

zip_adi_full_df.to_csv(OUT_FULL_CSV, index=False, encoding="utf-8-sig")
zip_adi_full_gdf.to_file(OUT_FULL_GPKG, driver="GPKG")
zip_adi_full_gdf.to_parquet(OUT_FULL_PARQUET)

print("saved full:")
print(OUT_FULL_CSV)
print(OUT_FULL_GPKG)
print(OUT_FULL_PARQUET)

saved full:
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_full.csv
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_full.gpkg
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_full.parquet


In [13]:
# =========================================================
# 14. analysis版を作成して保存
# =========================================================
zip_adi_analysis_gdf = zip_adi_gdf[
    zip_adi_gdf["ADI_raw"].notna()
].copy()

zip_adi_analysis_df = pd.DataFrame(zip_adi_analysis_gdf.drop(columns="geometry"))

zip_adi_analysis_df.to_csv(OUT_ANALYSIS_CSV, index=False, encoding="utf-8-sig")
zip_adi_analysis_gdf.to_file(OUT_ANALYSIS_GPKG, driver="GPKG")
zip_adi_analysis_gdf.to_parquet(OUT_ANALYSIS_PARQUET)

print("\nsaved analysis:")
print(OUT_ANALYSIS_CSV)
print(OUT_ANALYSIS_GPKG)
print(OUT_ANALYSIS_PARQUET)

print("analysis rows:", len(zip_adi_analysis_gdf))
print("analysis missing ADI_raw:", zip_adi_analysis_gdf["ADI_raw"].isna().sum())


saved analysis:
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_analysis.csv
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_analysis.gpkg
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/ziparea/zip_adi_analysis.parquet
analysis rows: 112980
analysis missing ADI_raw: 0
